In [1]:
from collections.abc import Iterable, Generator, Sequence
from dataclasses import dataclass
from math import tau, cos, ceil, floor

In [16]:
@dataclass
class Spec:
    sample_rate: int
    mark_frequency: int
    mark_num_periods: int
    space_frequency: int
    space_num_periods: int
    low: int
    high: int

    @classmethod
    def with_kcs(cls) -> 'Spec':
        return Spec(9600, 2400, 8, 1200, 4, -127, 127)


def samples_per_bit(sr: int, f: int, n: int) -> int:
    return (n * sr) // f


def to_nrz(bit: bool) -> int:
    return int(bit) * 2 - 1


def modulate_sample(a: float, cw: float, dw: float, t: float, dt: float) -> float:
    return a * cos(cw * t - dw * dt)


def interpolate(steps, data: Sequence[int]) -> Generator[(int, float), None, None]:
    n = steps * len(data)
    yield (0, 0)
    for i in range(1, n):
        index = int(ceil((i + 1) / steps)) - 1
        index_prev = int(ceil(i / steps)) - 1
        yield (i, (data[index_prev] + data[index]) / 2)


assert list(interpolate(2, list(range(0, 4)))) == [(0, 0), (1, 0), (2, 0.5), (3, 1), (4, 1.5), (5, 2), (6, 2.5), (7, 3)]


def integrate(data: Iterable[(int, float)]) -> Generator[(int, float), None, None]:
    m = 0.0
    for (i, d) in data:
        m += d
        yield (i, m)


assert list(integrate([(0, 0), (1, 3), (2, 2)])) == [(0, 0), (1, 3), (2, 5)]


def interpolate_and_integrate(steps: int, data: Sequence[int]) -> Generator[(int, float), None, None]:
    n = steps * len(data)
    m = 0.0
    yield (1, m)
    for i in range(2, n + 1):
        index = int(ceil(i / steps))
        index_prev = int(ceil((i - 1) / steps))
        m += (data[index_prev - 1] + data[index - 1]) / 2
        yield (i, m)


# pub fn modulate<I, S>(spec: &Spec<S>, data: I) -> impl Iterator<Item = S>
# where
#     I: Iterator<Item = bool>,
#     S: Copy + Into<f64> + FromF64Unchecked,
# {
#     // Amplitude Settings
#     let amplitude = (spec.high.into() - spec.low.into()) / 2.0;
# 
#     // Frequency Settings
#     // carrier_freq + delta_freq = 2400 Hz; carrier_freq - delta_freq = 1200 Hz
#     let carrier_freq = u32::midpoint(spec.mark_frequency, spec.space_frequency);
#     let delta_freq = spec.mark_frequency.abs_diff(spec.space_frequency) / 2;
#     let sample_rate = spec.sample_rate;
#     let carrier_omega = 2.0 * PI * (f64::from(carrier_freq) / f64::from(sample_rate));
#     let delta_omega = 2.0 * PI * (f64::from(delta_freq) / f64::from(sample_rate));
# 
#     // Integration settings
#     let steps = samples_per_bit(spec.sample_rate as usize, spec.mark_frequency as usize, spec.mark_num_periods);
# 
#     data.map(to_nrz)
#         .discrete_integral(steps)
#         .map(move |(i, m)| {
#             let y = modulate_sample(amplitude, carrier_omega, delta_omega, i as f64, m);
#             S::from_f64_unchecked(y)
#         })
# }
def modulate(spec: Spec, data: Iterable[bool]) -> Generator[int, None, None]:
    # Amplitude settings
    amplitude = (spec.high - spec.low) / 2

    # Frequency settings
    carrier_freq = (spec.mark_frequency + spec.space_frequency) // 2
    delta_freq = abs(spec.mark_frequency - spec.space_frequency) // 2
    sample_rate = spec.sample_rate
    carrier_omega = tau * (carrier_freq / sample_rate)
    delta_omega = tau * (delta_freq / sample_rate)

    steps = samples_per_bit(sample_rate, spec.mark_frequency, spec.mark_num_periods)

    nrz_data = (to_nrz(bit) for bit in data)
    interpolated_data = interpolate(steps, list(nrz_data))
    integrated_data = integrate(interpolated_data)
    for i, m in integrated_data:
        y = modulate_sample(amplitude, carrier_omega, delta_omega, float(i), m)
        yield int(y)


waveform = list(modulate(Spec.with_kcs(), [True]))
len(waveform)

32